# Xdawn-based EEG Quality Evaluation

This notebook provides a comprehensive analysis of EEG data quality and similarity metrics.

## 0. Path Setup

Add project source folder to Python path.

In [ ]:
import sys
from pathlib import Path

# Add src folder to Python path
project_root = Path.cwd().parent
src_path = project_root / 'src' / 'eeg_synthetic'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'✓ Path setup complete: {src_path}')

In [ ]:
import sys
from pathlib import Path

# Add src folder to Python path
project_root = Path.cwd().parent
src_path = project_root / 'src' / 'eeg_synthetic'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'✓ Path setup complete: {src_path}')

## 1. Setup and Data Loading

Import required libraries and load the EEG dataset for analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from scipy import signal

from data_loader import BCIAUTLoader, plot_normalized_arrays

In [ ]:
# Set up paths to the dataset
project_root = os.getcwd()
dataset_path = os.path.join(project_root, 'data')

# Initialize the data loader for BCI-AUT dataset
loader = BCIAUTLoader(dataset_path)

# Load training data for subject 3, session 3
# X_train_sess: EEG signals [trials, channels, time]
# y_train_sess: Labels for each trial
X_train_sess, y_train_sess, subjects_train_ids, session_train_ids = loader.get_data(
    subjects=[3], sessions=[3], modes=['Train']
)

# Load test data for the same subject and session
X_test_sess, y_test_sess, subjects_test_ids, session_test_ids = loader.get_data(
    subjects=[3], sessions=[3], modes=['Test']
)

# Visualize the normalized arrays to check data distribution
plot_normalized_arrays(X_train_sess, y_train_sess, X_test_sess, y_test_sess)

## 2. Synthetic Data Loading

Load and prepare synthetic EEG data for comparison with real data.

In [ ]:
def load_balanced_synthetic_limited(path_c0, path_c1, max_trials=1200):
    """
    Load balanced synthetic EEG data for both classes.
    
    Parameters:
    -----------
    path_c0 : str
        Path to class 0 synthetic data CSV file
    path_c1 : str
        Path to class 1 synthetic data CSV file
    max_trials : int
        Maximum number of trials to load per class
        
    Returns:
    --------
    X0_trial : torch.Tensor
        Class 0 EEG data with shape (trials, channels, time)
    y0_trial : torch.Tensor
        Class 0 labels
    X1_trial : torch.Tensor
        Class 1 EEG data with shape (trials, channels, time)
    y1_trial : torch.Tensor
        Class 1 labels
    """
    # Each trial has 8 channels, so multiply by 8 to get total rows
    rows_to_read = max_trials * 8

    def get_data_from_csv(path):
        """Helper function to read CSV and extract EEG data."""
        if not os.path.exists(path):
            print(f"ERROR: File not found - {path}")
            return None, None
        
        # Read CSV file with specified number of rows
        df = pd.read_csv(path, nrows=rows_to_read)
        
        # Extract time series columns (e.g., Time_0, Time_1, ..., Time_127)
        time_cols = [col for col in df.columns if col.startswith('Time')]
        X = df[time_cols].values
        
        # Extract condition labels (0 or 1)
        y = df['Condition'].values
        return X, y
    
    # Load data for both classes
    X0_rows, y0_rows = get_data_from_csv(path_c0)
    X1_rows, y1_rows = get_data_from_csv(path_c1)
    
    # Reshape from flat format to [trials, channels=8, time=128]
    X0_trial = torch.tensor(X0_rows).view(-1, 8, 128)
    y0_trial = torch.tensor(y0_rows[::8])  # Take every 8th label (one per trial)
    
    X1_trial = torch.tensor(X1_rows).view(-1, 8, 128)
    y1_trial = torch.tensor(y1_rows[::8])

    return X0_trial, y0_trial, X1_trial, y1_trial


# Paths to synthetic data CSV files
file_c0 = 'generated_samples/gen_data_c0_s3_sess3.csv'
file_c1 = 'generated_samples/gen_data_c1_s3_sess3.csv'

# Load up to 8600 trials per class
X0_syn, y0_syn, X1_syn, y1_syn = load_balanced_synthetic_limited(
    file_c0, file_c1, max_trials=8600
)

# Combine both classes into single tensors
X_syn_final = torch.cat([X0_syn, X1_syn], dim=0)
y_syn_final = torch.cat([y0_syn, y1_syn], dim=0)

## 3. Data Preprocessing

Apply filtering and baseline correction to prepare data for analysis.

In [ ]:
def mne_style_iir_filter(data, fs=128):
    """
    Apply bandpass IIR filter to EEG data.
    
    Parameters:
    -----------
    data : np.ndarray
        EEG data to filter
    fs : int
        Sampling frequency in Hz
        
    Returns:
    --------
    np.ndarray
        Filtered EEG data
    """
    # Define bandpass filter parameters
    low_cutoff = 0.1   # High-pass at 0.1 Hz (removes slow drifts)
    high_cutoff = 15.0  # Low-pass at 15 Hz (removes high-frequency noise)
    order = 4           # 4th order Butterworth filter
    
    # Design Butterworth bandpass filter
    b, a = signal.butter(order, [low_cutoff, high_cutoff], btype='bandpass', fs=fs)
    
    # Apply zero-phase filtering (forward and backward pass)
    return signal.filtfilt(b, a, data, axis=-1)


def apply_final_alignment(x_tensor):
    """
    Apply filtering and baseline correction to EEG data.
    
    Parameters:
    -----------
    x_tensor : torch.Tensor
        Input EEG data with shape (trials, channels, time)
        
    Returns:
    --------
    torch.Tensor
        Preprocessed EEG data with filtering and baseline correction applied
    """
    # Convert to numpy for signal processing
    x_np = x_tensor.detach().cpu().numpy()
    
    # Step 1: Apply bandpass filter (0.1-15 Hz)
    x_filtered = mne_style_iir_filter(x_np, fs=128)
    
    # Step 2: Baseline correction using first 26 samples (~200ms at 128 Hz)
    # This removes DC offset and normalizes the baseline period
    baseline_period = x_filtered[:, :, :26]
    baseline_mean = np.mean(baseline_period, axis=-1, keepdims=True)
    x_aligned = x_filtered - baseline_mean
    
    # Convert back to PyTorch tensor
    return torch.from_numpy(x_aligned.copy()).float()

In [ ]:
# ============================================================================
# PREPROCESS SYNTHETIC DATA
# ============================================================================

# Apply filtering and baseline correction to synthetic data
X_syn_ready = apply_final_alignment(X_syn_final)
X_syn_filtered = torch.from_numpy(X_syn_ready.detach().cpu().numpy().copy()).float()

# Z-score normalization (trial-wise)
# This standardizes each trial independently to have mean=0 and std=1
mu_syn = X_syn_filtered.mean(dim=(1, 2), keepdim=True)
std_syn = X_syn_filtered.std(dim=(1, 2), keepdim=True)
X_syn_filtered = (X_syn_filtered - mu_syn) / std_syn

# Permute dimensions from [trials, channels, time] to [trials, time, channels]
X_grouped_perm = X_syn_filtered.permute(0, 2, 1)

# Separate by class: 0 = "win" (non-target), 1 = "loss" (target)
syn_win = X_grouped_perm[y_syn_final == 0.0]
syn_loss = X_grouped_perm[y_syn_final == 1.0]

# ============================================================================
# PREPROCESS REAL DATA (TRAIN AND TEST)
# ============================================================================

# Convert real data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_sess).float()
y_train_tensor = torch.tensor(y_train_sess).float()
X_test_sess_tensor = torch.tensor(X_test_sess).float()
y_test_sess_tensor = torch.tensor(y_test_sess).float()

# Apply the same filtering and baseline correction
X_train_tensor = apply_final_alignment(X_train_tensor)
X_test_sess_tensor = apply_final_alignment(X_test_sess_tensor)

# Z-score normalization for training data
mu_ref = X_train_tensor.mean(dim=(1, 2), keepdim=True)
std_ref = X_train_tensor.std(dim=(1, 2), keepdim=True)
X_train_real_z = (X_train_tensor - mu_ref) / std_ref

# Z-score normalization for test data
mu_ref_test = X_test_sess_tensor.mean(dim=(1, 2), keepdim=True)
std_ref_test = X_test_sess_tensor.std(dim=(1, 2), keepdim=True)
X_test_real_z = (X_test_sess_tensor - mu_ref_test) / std_ref_test

# Permute to [trials, time, channels] format
X_train_real_ready = X_train_real_z.permute(0, 2, 1)
X_test_real_ready = X_test_real_z.permute(0, 2, 1)

print(f"Real Data: Train {X_train_real_ready.shape}, Test {X_test_real_ready.shape}")

# Separate real data by class
real_train_0 = X_train_real_ready[y_train_tensor == 0]  # Non-target trials (train)
real_test_0 = X_test_real_ready[y_test_sess_tensor == 0]  # Non-target trials (test)
real_train_1 = X_train_real_ready[y_train_tensor == 1]  # Target trials (train)
real_test_1 = X_test_real_ready[y_test_sess_tensor == 1]  # Target trials (test)

## 4. Xdawn-based Fréchet Distance Evaluation

**What is Xdawn?**
Xdawn is a spatial filtering technique specifically designed for Event-Related Potentials (ERPs). It enhances the signal-to-noise ratio by finding optimal spatial filters that maximize the difference between target and non-target responses.

**Why use Xdawn for evaluation?**
- Extracts task-relevant features from EEG data
- Reduces dimensionality while preserving discriminative information
- Provides a biologically-motivated feature space for comparison

**Evaluation approach:**
1. Train Xdawn on real training data
2. Transform real and synthetic data using the fitted Xdawn filters
3. Compute Fréchet distance between feature distributions
4. Lower FID = synthetic data better matches real data distribution

In [ ]:
import mne
from mne.preprocessing import Xdawn
from scipy.linalg import sqrtm


def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """
    Compute Fréchet Inception Distance (FID) between two Gaussian distributions.
    
    The FID measures the distance between two multivariate Gaussians:
    FID = ||mu1 - mu2||^2 + Tr(sigma1 + sigma2 - 2*sqrt(sigma1*sigma2))
    
    Lower FID indicates more similar distributions (better synthetic data quality).
    
    Parameters:
    -----------
    mu1, mu2 : np.ndarray
        Mean vectors of the two distributions
    sigma1, sigma2 : np.ndarray
        Covariance matrices of the two distributions
    eps : float
        Small constant for numerical stability
        
    Returns:
    --------
    float
        Fréchet distance between the two distributions
    """
    # Calculate difference between means
    diff = mu1 - mu2
    
    # Calculate the square root of the product of covariance matrices
    covmean = sqrtm(sigma1.dot(sigma2))
    
    # Handle numerical instability
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = sqrtm((sigma1 + offset).dot(sigma2 + offset))
    
    # Handle complex numbers from matrix square root
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    # Compute FID using the formula
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)


def get_xdawn_features(X_np, xdawn_model, info):
    """
    Apply Xdawn spatial filtering and extract features.
    
    Parameters:
    -----------
    X_np : np.ndarray
        EEG data with shape (trials, channels, time)
    xdawn_model : Xdawn
        Fitted Xdawn model
    info : mne.Info
        MNE info object with channel information
        
    Returns:
    --------
    np.ndarray
        Flattened Xdawn features with shape (trials, features)
    """
    temp_epochs = mne.EpochsArray(X_np, info, verbose=False)
    X_filt = xdawn_model.transform(temp_epochs)
    return X_filt.reshape(X_filt.shape[0], -1)


def get_stats(feat):
    """
    Calculate mean and covariance of features.
    
    Parameters:
    -----------
    feat : np.ndarray
        Feature matrix with shape (samples, features)
        
    Returns:
    --------
    mu : np.ndarray
        Mean vector
    sigma : np.ndarray
        Covariance matrix
    """
    mu = np.mean(feat, axis=0)
    sigma = np.cov(feat, rowvar=False)
    return mu, sigma

ch_names = ['C3', 'Cz', 'C4', 'CPz', 'P3', 'Pz', 'P4', 'POz']
info = mne.create_info(ch_names=ch_names, sfreq=128, ch_types='eeg')

X_train_np = X_train_real_z.cpu().numpy() 
y_train_np = y_train_tensor.cpu().numpy().astype(int)

events = np.zeros((len(y_train_np), 3), dtype=int)
events[:, 0] = np.arange(len(y_train_np))
events[:, 2] = y_train_np           

epochs_train = mne.EpochsArray(X_train_np, info, events=events, 
                               event_id={'Non-Target': 0, 'Target': 1}, verbose=False)

# --- TRAINING XDAWN ---
n_components = 4
xdawn = Xdawn(n_components=n_components)
xdawn.fit(epochs_train)

# --- Feature Extraction with xDAWN ---
feat_real_train_1 = get_xdawn_features(real_train_1.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_real_test_1  = get_xdawn_features(real_test_1.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_syn_1        = get_xdawn_features(syn_loss.permute(0, 2, 1).cpu().numpy(), xdawn, info)
# --- Average and Covariance ---
def get_stats(feat):
    mu = np.mean(feat, axis=0)
    sigma = np.cov(feat, rowvar=False)
    return mu, sigma

mu_tr, sig_tr = get_stats(feat_real_train_1)
mu_te, sig_te = get_stats(feat_real_test_1)
mu_sy, sig_sy = get_stats(feat_syn_1)

# --- Compute FID ---
fid_tr_te = calculate_frechet_distance(mu_tr, sig_tr, mu_te, sig_te)
fid_tr_sy = calculate_frechet_distance(mu_tr, sig_tr, mu_sy, sig_sy)
fid_sy_te = calculate_frechet_distance(mu_sy, sig_sy, mu_te, sig_te)

print(f"\n" + "="*50)
print(f"{'FID on xDAWN features (Class 1)':^50}")
print("="*50)
print(f"1. Real Train vs Real Test:   {fid_tr_te:.4f}")
print(f"2. Real Train vs Synthetic:   {fid_tr_sy:.4f}")
print(f"3. Synthetic  vs Real Test:   {fid_sy_te:.4f}")
print("-" * 50)

# --- REPEAT FOR CLASS 0 ---
feat_real_train_0 = get_xdawn_features(real_train_0.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_real_test_0  = get_xdawn_features(real_test_0.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_syn_0        = get_xdawn_features(syn_win.permute(0, 2, 1).cpu().numpy(), xdawn, info)

mu_tr0, sig_tr0 = get_stats(feat_real_train_0)
mu_te0, sig_te0 = get_stats(feat_real_test_0)
mu_sy0, sig_sy0 = get_stats(feat_syn_0)

fid_tr_te0 = calculate_frechet_distance(mu_tr0, sig_tr0, mu_te0, sig_te0)
fid_tr_sy0 = calculate_frechet_distance(mu_tr0, sig_tr0, mu_sy0, sig_sy0)
fid_sy_te0 = calculate_frechet_distance(mu_sy0, sig_sy0, mu_te0, sig_te0)

print(f"\n" + "="*50)
print(f"{'FID on xDAWN features (Class 0)':^50}")
print("="*50)
print(f"1. Real Train vs Real Test:   {fid_tr_te0:.4f}")
print(f"2. Real Train vs Synthetic:   {fid_tr_sy0:.4f}")
print(f"3. Synthetic  vs Real Test:   {fid_sy_te0:.4f}")
print("-" * 50)

In [ ]:
# ============================================================================
# SETUP MNE INFO OBJECT
# ============================================================================

# Define the 8 EEG channel names used in this experiment
ch_names = ['C3', 'Cz', 'C4', 'CPz', 'P3', 'Pz', 'P4', 'POz']

# Create MNE info object with channel metadata
info = mne.create_info(ch_names=ch_names, sfreq=128, ch_types='eeg')

# ============================================================================
# PREPARE DATA FOR XDAWN TRAINING
# ============================================================================

# Convert training data to numpy arrays
X_train_np = X_train_real_z.cpu().numpy()
y_train_np = y_train_tensor.cpu().numpy().astype(int)

# Create events array for MNE (required format: [sample_index, 0, event_id])
events = np.zeros((len(y_train_np), 3), dtype=int)
events[:, 0] = np.arange(len(y_train_np))  # Sample indices
events[:, 2] = y_train_np                   # Event IDs (0 or 1)

# Create MNE Epochs object for Xdawn training
epochs_train = mne.EpochsArray(
    X_train_np, info, events=events,
    event_id={'Non-Target': 0, 'Target': 1},  # Map labels to event names
    verbose=False
)

# ============================================================================
# TRAIN XDAWN SPATIAL FILTERS
# ============================================================================

# Number of spatial components to extract (4 is typical for P300 paradigms)
n_components = 4

# Initialize and fit Xdawn on real training data
xdawn = Xdawn(n_components=n_components)
xdawn.fit(epochs_train)

# ============================================================================
# EVALUATE CLASS 1 (TARGET/LOSS TRIALS)
# ============================================================================

# Extract Xdawn features for Class 1 data
# Note: permute to [trials, channels, time] format expected by MNE
feat_real_train_1 = get_xdawn_features(real_train_1.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_real_test_1 = get_xdawn_features(real_test_1.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_syn_1 = get_xdawn_features(syn_loss.permute(0, 2, 1).cpu().numpy(), xdawn, info)

# Calculate mean and covariance for each dataset
mu_tr, sig_tr = get_stats(feat_real_train_1)
mu_te, sig_te = get_stats(feat_real_test_1)
mu_sy, sig_sy = get_stats(feat_syn_1)

# Compute Fréchet distances between distributions
fid_tr_te = calculate_frechet_distance(mu_tr, sig_tr, mu_te, sig_te)  # Baseline: train vs test
fid_tr_sy = calculate_frechet_distance(mu_tr, sig_tr, mu_sy, sig_sy)  # Main: train vs synthetic
fid_sy_te = calculate_frechet_distance(mu_sy, sig_sy, mu_te, sig_te)  # Cross: synthetic vs test

# Display results for Class 1
print(f"\n" + "="*50)
print(f"{'FID on Xdawn features (Class 1 - Target)':^50}")
print("="*50)
print(f"1. Real Train vs Real Test:   {fid_tr_te:.4f}  (baseline)")
print(f"2. Real Train vs Synthetic:   {fid_tr_sy:.4f}  (quality metric)")
print(f"3. Synthetic  vs Real Test:   {fid_sy_te:.4f}  (generalization)")
print("-" * 50)

# ============================================================================
# EVALUATE CLASS 0 (NON-TARGET/WIN TRIALS)
# ============================================================================

# Extract Xdawn features for Class 0 data
feat_real_train_0 = get_xdawn_features(real_train_0.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_real_test_0 = get_xdawn_features(real_test_0.permute(0, 2, 1).cpu().numpy(), xdawn, info)
feat_syn_0 = get_xdawn_features(syn_win.permute(0, 2, 1).cpu().numpy(), xdawn, info)

# Calculate statistics
mu_tr0, sig_tr0 = get_stats(feat_real_train_0)
mu_te0, sig_te0 = get_stats(feat_real_test_0)
mu_sy0, sig_sy0 = get_stats(feat_syn_0)

# Compute Fréchet distances
fid_tr_te0 = calculate_frechet_distance(mu_tr0, sig_tr0, mu_te0, sig_te0)
fid_tr_sy0 = calculate_frechet_distance(mu_tr0, sig_tr0, mu_sy0, sig_sy0)
fid_sy_te0 = calculate_frechet_distance(mu_sy0, sig_sy0, mu_te0, sig_te0)

# Display results for Class 0
print(f"\n" + "="*50)
print(f"{'FID on Xdawn features (Class 0 - Non-Target)':^50}")
print("="*50)
print(f"1. Real Train vs Real Test:   {fid_tr_te0:.4f}  (baseline)")
print(f"2. Real Train vs Synthetic:   {fid_tr_sy0:.4f}  (quality metric)")
print(f"3. Synthetic  vs Real Test:   {fid_sy_te0:.4f}  (generalization)")
print("-" * 50)